<a href="https://colab.research.google.com/github/Pauaza/PBL_Sistem-Cerdas-Rencana-Riset/blob/uts-sbp/certainty_factor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Import library
import pandas as pd
import re

In [14]:
# Membaca dataset
df = pd.read_csv('/content/drive/MyDrive/Kuliah/Semester 6/sbp/UTS/master_dataset_dosen.csv')

# Menampilkan data awal
print("Preview Data Awal:")
print(df.head())

Preview Data Awal:
   id_dosen_master                            nama_dosen lab_code  \
0                1               Ade Ismail S.Kom., M.TI      NCS   
1                2  Adevian Fairuz Pratama, S.S.T, M.Eng      MMT   
2                3  Agung Nugroho Pramudhita, S.T., M.T.    InLET   
3                4         Ahmadi Yuli Ananta, ST., M.M.       BA   
4                5    Anugrah Nur Rahmanto, S.Sn., M.Ds.      MMT   

                                          lab_name  is_kalab  \
0                       Network and Cyber Security       0.0   
1                         Multimedia & Mobile Tech       0.0   
2  Information and Learning Engineering Technology       0.0   
3                               Business Analytics       0.0   
4                         Multimedia & Mobile Tech       0.0   

                                    judul_penelitian  \
0  monitoring suhu dan gas amonia untuk kandang c...   
1  machine learning for vocational undergraduate ...   
2  integratin

In [15]:
# Mendefinisikan Fungsi Certainty Factor
def calculate_cf(input_mhs, profil_dosen):
    # Preprocessing: Tokenisasi dan pembersihan kata
    words_input = set(re.findall(r'\w+', input_mhs.lower()))
    words_dosen = set(re.findall(r'\w+', profil_dosen.lower()))

    # Menghitung irisan kata (Evidence)
    intersection = words_input.intersection(words_dosen)
    match_count = len(intersection)

    # MB: Measure of Belief (Kekuatan bukti yang mendukung)
    # Dihitung dari rasio kata yang cocok terhadap total kata input
    mb = match_count / len(words_input) if len(words_input) > 0 else 0

    # MD: Measure of Disbelief (Ketidakyakinan / Noise)
    # Kita tetapkan 0.05 sebagai ambang batas ketidakpastian data teks
    md = 0.05 if mb > 0 else 0

    # Rumus CF
    cf_score = mb - md
    return max(0, cf_score)

In [43]:
# Proses Pencarian Rekomendasi

# --- INPUT MAHASISWA ---
topik_mhs = "Business Intelligence"
deskripsi_ide = "Penerapan dashboard untuk monitoring penjualan ritel dan prediksi stok menggunakan data historis"

# Gabungkan input
input_full = topik_mhs + " " + deskripsi_ide

# Hitung Skor CF
# Gunakan str(x) untuk memastikan input ke fungsi adalah string
df['cf_score'] = df['judul_penelitian'].apply(lambda x: calculate_cf(input_full, str(x)))

# Mengambil 3 Dosen Terbaik
rekomendasi_dosen = df.sort_values(by='cf_score', ascending=False).head(3)

print(f"=== HASIL ANALISIS SISTEM (CERTAINTY FACTOR) ===\n")

print(f"Topik: {topik_mhs}")
print("-" * 50)
print(f"Deskripsi Ide: {deskripsi_ide}")
print("-" * 50)

# 1. PERHITUNGAN CERTAINTY FACTOR
input_full = topik_mhs + " " + deskripsi_ide
df['cf_score'] = df['judul_penelitian'].apply(lambda x: calculate_cf(input_full, str(x)))

# Ambil data dosen terbaik (Top 3)
top_dosen = df.sort_values(by='cf_score', ascending=False).head(3)

# BAGIAN 1: REKOMENDASI DOSEN PEMBIMBING
print(f"1. REKOMENDASI DOSEN PEMBIMBING")
for i, (index, row) in enumerate(top_dosen.iterrows()):
    print(f"Peringkat {i+1}: {row['nama_dosen']}")
    print(f"   - Lab           : {row['lab_name']}")
    print(f"   - Skor Kepastian: {row['cf_score']:.2f}")

# BAGIAN 2: SARAN JUDUL SKRIPSI (BERDASARKAN TOPIK & KEAHLIAN)
print(f"\n2. SARAN JUDUL SKRIPSI")

# Mengambil 'kata kunci' dari judul penelitian dosen peringkat 1 untuk diramu
judul_ref = str(top_dosen.iloc[0]['judul_penelitian']).split('|')[0].strip().split()
keahlian_inti = " ".join(judul_ref[:2]) # Mengambil 2 kata pertama sebagai representasi metode

# List Template Judul Kreatif
saran_judul = [
    f"Pengembangan Sistem {topik_mhs} untuk {deskripsi_ide} Berbasis {keahlian_inti.title()}",
    f"Implementasi Metode {keahlian_inti.title()} pada {topik_mhs} untuk Optimasi {deskripsi_ide.split()[-1]}",
    f"Rancang Bangun Dashboard {topik_mhs}: Studi Kasus {deskripsi_ide.split()[-1].title()} Menggunakan Pendekatan {keahlian_inti.title()}",
    f"Analisis Prediktif {topik_mhs} dalam {deskripsi_ide} untuk Mendukung Keputusan Bisnis"
]

for i, judul in enumerate(saran_judul):
    # Membersihkan judul agar rapi (title case)
    print(f"Saran {i+1} : {judul}")

=== HASIL ANALISIS SISTEM (CERTAINTY FACTOR) ===

Topik: Business Intelligence
--------------------------------------------------
Deskripsi Ide: Penerapan dashboard untuk monitoring penjualan ritel dan prediksi stok menggunakan data historis
--------------------------------------------------
1. REKOMENDASI DOSEN PEMBIMBING
Peringkat 1: Candra Bella Vista, S.Kom., MT.
   - Lab           : Business Analytics
   - Skor Kepastian: 0.38
Peringkat 2: Mungki Astiningrum, ST., M.Kom.
   - Lab           : Intelligent Vision and Smart System
   - Skor Kepastian: 0.38
Peringkat 3: Budi Harijanto, ST., M.MKom.
   - Lab           : Information and Learning Engineering Technology
   - Skor Kepastian: 0.31

2. SARAN JUDUL SKRIPSI
Saran 1 : Pengembangan Sistem Business Intelligence untuk Penerapan dashboard untuk monitoring penjualan ritel dan prediksi stok menggunakan data historis Berbasis Pengembangan Chatbot
Saran 2 : Implementasi Metode Pengembangan Chatbot pada Business Intelligence untuk Optima